# 濃縮影片觀看行為調查分析
**研究問題：** 18–35 歲受眾在觀看完濃縮影片後，會產生什麼後續行為？這些行為對正版影視內容是促進還是侵蝕？

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
# Set seaborn style FIRST — it resets rcParams, so font must come after
sns.set_style('whitegrid')

# Chinese font setup (must be after sns.set_style)
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'STHeiti', 'Heiti TC']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.unicode_minus'] = False

_found = fm.findfont('Arial Unicode MS', fallback_to_default=False)
print(f'✅ 中文字型：Arial Unicode MS → {_found}')

PALETTE = sns.color_palette('Set2')

## 1. 資料載入與清理

In [ ]:
RAW_PATH = '../-AIAGENT/data.csv'

df_raw = pd.read_csv(RAW_PATH)
print(f'原始資料：{df_raw.shape[0]} 列 × {df_raw.shape[1]} 欄')
df_raw.head(3)

In [ ]:
# 只保留通過篩選（有觀看濃縮影片）的受訪者
df = df_raw[
    (df_raw['early_exit'].isna()) &
    (df_raw['screening_watched_condensed'] == '是')
].copy().reset_index(drop=True)

print(f'有效受訪者：{len(df)} 人')
print(f'\n欄位一覽：')
print(df.dtypes)

## 2. 樣本基本特徵

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('樣本基本特徵', fontsize=16, fontweight='bold', y=1.02)

# 性別
gender_counts = df['gender'].value_counts()
axes[0].pie(gender_counts.values, labels=gender_counts.index,
            autopct='%1.1f%%', colors=PALETTE, startangle=90)
axes[0].set_title('性別分佈')

# 年齡
age_order = ['18 歲以下', '18–25 歲', '26–35 歲', '36–45 歲', '46 歲以上']
age_counts = df['age'].value_counts().reindex(
    [a for a in age_order if a in df['age'].values], fill_value=0
)
axes[1].bar(range(len(age_counts)), age_counts.values,
            color=sns.color_palette('Set2', len(age_counts)))
axes[1].set_xticks(range(len(age_counts)))
axes[1].set_xticklabels(age_counts.index, rotation=30, ha='right')
axes[1].set_title('年齡分佈')
axes[1].set_ylabel('人數')
for i, v in enumerate(age_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=11)

# NTU 學生
ntu_counts = df['ntu_student'].value_counts()
axes[2].pie(ntu_counts.values, labels=['NTU 學生' if x=='是' else '非 NTU' for x in ntu_counts.index],
            autopct='%1.1f%%', colors=['#4CAF50', '#FF7043'], startangle=90)
axes[2].set_title('是否為 NTU 學生')

plt.tight_layout()
plt.savefig('01_demographics.png', bbox_inches='tight')
plt.show()

## 3. 喜愛的濃縮影片類型

In [ ]:
def split_multi(series, sep='|'):
    """Multi-select column → flat list of all choices."""
    items = []
    for val in series.dropna():
        items.extend([x.strip() for x in str(val).split(sep)])
    return items

types_raw = split_multi(df['favorite_condensed_types'])
# Exclude 'other' freetext
types_clean = [t for t in types_raw if '其他' not in t]
type_counts = Counter(types_clean)

fig, ax = plt.subplots(figsize=(10, 5))
labels = list(type_counts.keys())
vals = list(type_counts.values())
sorted_idx = np.argsort(vals)[::-1]
ax.bar([labels[i] for i in sorted_idx], [vals[i] for i in sorted_idx],
       color=sns.color_palette('Set2', len(labels)))
ax.set_title('喜愛的濃縮影片類型（複選）', fontsize=14)
ax.set_ylabel('提及次數')
ax.set_xlabel('類型')
for i, v in enumerate([vals[j] for j in sorted_idx]):
    ax.text(i, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('02_fav_types.png', bbox_inches='tight')
plt.show()

## 4. 觀看濃縮影片的動機

In [ ]:
motiv_raw = split_multi(df['watch_motivation'])
motiv_clean = [m for m in motiv_raw if '其他' not in m and '視作品種類而定' not in m]
motiv_counts = Counter(motiv_clean)

# Short labels for readability
label_map = {
    '想先篩選原版是否值得花時間觀看': '篩選原版',
    '沒有時間看原版，但想了解劇情': '沒時間看原版',
    '演算法推薦，被動接觸': '演算法推薦',
    '原版已看過，想重溫重點': '重溫重點',
    '視作品種類而定': '視作品而定',
}
mc_items = sorted(motiv_counts.items(), key=lambda x: -x[1])
labels_m = [label_map.get(k, k) for k, v in mc_items]
vals_m = [v for k, v in mc_items]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels_m, vals_m, color=sns.color_palette('Set3', len(labels_m)))
ax.set_title('觀看濃縮影片的動機（複選）', fontsize=14)
ax.set_xlabel('提及次數')
ax.invert_yaxis()
for bar, v in zip(bars, vals_m):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2, str(v), va='center', fontsize=11)
plt.tight_layout()
plt.savefig('03_watch_motivation.png', bbox_inches='tight')
plt.show()

## 5. 觀看後的後續行為（核心分析）

In [ ]:
actions_raw = split_multi(df['actions_after_condensed'])
actions_clean = [a for a in actions_raw if '其他' not in a]
action_counts = Counter(actions_clean)

# Categorise each action as 'promote', 'erode', or 'neutral'
action_category = {
    '在已訂閱的串流平台看完完整作品': 'promote',
    '在已訂閱的串流平台觀看作品部分內容': 'promote',
    '透過正版免費管道觀看原版（如電視播映、YouTube 官方頻道）': 'promote',
    '將原版加入待看清單': 'promote',
    '搜尋原版作品的相關資訊（如評價、演員）': 'neutral',
    '至 YouTube 等平台找其他此作品的濃縮影片觀看': 'erode',
    '對原版作品留下印象，但沒有進一步行動': 'neutral',
    '透過非正式授權管道觀看原版（如盜版網站、非授權影片）': 'erode',
}

ac_items = sorted(action_counts.items(), key=lambda x: -x[1])
labels_a = [k for k, v in ac_items]
vals_a = [v for k, v in ac_items]
colors_a = [
    '#4CAF50' if action_category.get(k) == 'promote' else
    '#F44336' if action_category.get(k) == 'erode' else '#9E9E9E'
    for k in labels_a
]

short_labels = {
    '在已訂閱的串流平台看完完整作品': '串流平台看完整版',
    '在已訂閱的串流平台觀看作品部分內容': '串流平台看部分',
    '透過正版免費管道觀看原版（如電視播映、YouTube 官方頻道）': '正版免費管道',
    '將原版加入待看清單': '加入待看清單',
    '搜尋原版作品的相關資訊（如評價、演員）': '搜尋相關資訊',
    '至 YouTube 等平台找其他此作品的濃縮影片觀看': '再找更多濃縮片',
    '對原版作品留下印象，但沒有進一步行動': '留下印象/無行動',
    '透過非正式授權管道觀看原版（如盜版網站、非授權影片）': '盜版管道觀看',
}

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh([short_labels.get(l, l) for l in labels_a], vals_a, color=colors_a)
ax.set_title('觀看濃縮影片後的後續行為', fontsize=14)
ax.set_xlabel('提及次數')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4CAF50', label='促進正版消費'),
    Patch(facecolor='#F44336', label='侵蝕正版消費'),
    Patch(facecolor='#9E9E9E', label='中立行為'),
]
ax.legend(handles=legend_elements, loc='lower right')
for bar, v in zip(bars, vals_a):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2, str(v), va='center', fontsize=11)

plt.tight_layout()
plt.savefig('04_actions_after.png', bbox_inches='tight')
plt.show()

## 6. 整體影響：促進 vs 侵蝕

In [ ]:
impact_counts = df['overall_impact_on_original'].value_counts()

impact_color_map = {
    '讓我看了更多原版作品': '#4CAF50',
    '讓我看了更少原版作品': '#F44336',
    '沒有明顯影響': '#9E9E9E',
    '視作品而定，沒有固定傾向': '#2196F3',
}
colors_i = [impact_color_map.get(k, '#CCCCCC') for k in impact_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie
wedges, texts, autotexts = axes[0].pie(
    impact_counts.values, labels=impact_counts.index,
    autopct='%1.1f%%', colors=colors_i, startangle=90,
    textprops={'fontsize': 10}
)
axes[0].set_title('濃縮影片對原版觀看的整體影響', fontsize=13)

# Bar
axes[1].bar(range(len(impact_counts)), impact_counts.values, color=colors_i)
axes[1].set_xticks(range(len(impact_counts)))
axes[1].set_xticklabels(impact_counts.index, rotation=20, ha='right', fontsize=10)
axes[1].set_ylabel('人數')
axes[1].set_title('整體影響 — 人數分佈', fontsize=13)
for i, v in enumerate(impact_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('05_overall_impact.png', bbox_inches='tight')
plt.show()

print('\n整體影響摘要：')
for k, v in impact_counts.items():
    print(f'  {k}: {v} 人 ({v/len(df)*100:.1f}%)')

## 7. 整體影響 × 性別 / NTU 交叉分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, title in zip(axes,
                           ['gender', 'ntu_student'],
                           ['整體影響 × 性別', '整體影響 × NTU 學生']):
    ct = pd.crosstab(df[col], df['overall_impact_on_original'])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=ax, colormap='Set2', width=0.7)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('百分比 (%)')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title='影響', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('06_impact_cross.png', bbox_inches='tight')
plt.show()

## 8. 「不採取行動」的原因（低努力組）

In [ ]:
low_raw = split_multi(df['low_effort_no_action_reasons'])
low_clean = [l for l in low_raw if l]
low_counts = Counter(low_clean)

low_label_map = {
    '劇情已透過濃縮版完整了解，不需要再看原版': '劇情已了解',
    '看完濃縮版後，對原版的興趣或期待感降低': '興趣降低',
    '有興趣但沒有足夠的時間': '沒時間',
    '有興趣但原版費用超出預算': '費用超出預算',
    '視影視作品種類而異': '視作品種類而異',
}

lc_items = sorted(low_counts.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(11, 5))
labels_l = [low_label_map.get(k, k) for k, v in lc_items]
vals_l = [v for k, v in lc_items]
bars = ax.bar(labels_l, vals_l, color=sns.color_palette('Oranges_r', len(labels_l)))
ax.set_title('不進一步行動的原因（複選）', fontsize=14)
ax.set_ylabel('提及次數')
ax.set_xticklabels(labels_l, rotation=20, ha='right')
for bar, v in zip(bars, vals_l):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('07_low_effort_reasons.png', bbox_inches='tight')
plt.show()

## 9. 實際去看原版的原因（高努力組）

In [ ]:
high_raw = split_multi(df['high_effort_watch_reasons'])
high_clean = [h for h in high_raw if h]
high_counts = Counter(high_clean)

high_label_map = {
    '被情緒或畫面氛圍吸引，想親身感受': '情緒/畫面吸引',
    '濃縮版確認了該作品符合我的喜好': '確認符合喜好',
    '劇情沒有完整交代，讓我想自己看完': '劇情未完整',
    '演員或製作陣容吸引我': '演員/製作吸引',
    '看到網友熱烈討論，想加入話題': '網路討論熱烈',
    '親友的分享或推薦讓我更想看': '親友推薦',
}

hc_items = sorted(high_counts.items(), key=lambda x: -x[1])
fig, ax = plt.subplots(figsize=(11, 5))
labels_h = [high_label_map.get(k, k) for k, v in hc_items]
vals_h = [v for k, v in hc_items]
bars = ax.bar(labels_h, vals_h, color=sns.color_palette('Greens_r', len(labels_h)))
ax.set_title('實際觀看原版的原因（複選）', fontsize=14)
ax.set_ylabel('提及次數')
ax.set_xticklabels(labels_h, rotation=20, ha='right')
for bar, v in zip(bars, vals_h):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('08_high_effort_reasons.png', bbox_inches='tight')
plt.show()

## 10. 觀看原版的頻率（高努力組）

In [ ]:
freq_order = ['幾乎每次', '經常', '偶爾', '幾乎不']
freq_counts = df['high_effort_watch_frequency'].value_counts().reindex(
    [f for f in freq_order if f in df['high_effort_watch_frequency'].values], fill_value=0
)

fig, ax = plt.subplots(figsize=(8, 5))
freq_colors = ['#388E3C', '#66BB6A', '#FFA726', '#EF5350']
bars = ax.bar(freq_counts.index, freq_counts.values,
              color=freq_colors[:len(freq_counts)])
ax.set_title('看了濃縮版後實際去看原版的頻率', fontsize=13)
ax.set_ylabel('人數')
for bar, v in zip(bars, freq_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, str(v), ha='center', fontsize=12)
plt.tight_layout()
plt.savefig('09_watch_freq.png', bbox_inches='tight')
plt.show()

## 11. 濃縮影片發現新作品的管道

In [ ]:
disc_raw = split_multi(df['discovered_via_condensed'])
disc_clean = [d for d in disc_raw if d]
disc_counts = Counter(disc_clean)

disc_label_map = {
    '完全不知道的新作品': '全新作品',
    '原本不會主動選擇的類型（如恐怖片、紀錄片）': '原本不選的類型',
    '原本不會主動選擇的國家或語言的作品（如韓劇、日劇）': '原本不選的語言/地區',
    '以上皆有': '以上皆有',
    '從未發生過，濃縮影片沒有擴展我的影視範圍': '從未發現新作品',
}

dc_items = sorted(disc_counts.items(), key=lambda x: -x[1])
fig, ax = plt.subplots(figsize=(11, 5))
labels_d = [disc_label_map.get(k, k) for k, v in dc_items]
vals_d = [v for k, v in dc_items]
bars = ax.bar(labels_d, vals_d, color=sns.color_palette('Purples_r', len(labels_d)))
ax.set_title('濃縮影片幫助發現的新內容類型', fontsize=14)
ax.set_ylabel('提及次數')
ax.set_xticklabels(labels_d, rotation=20, ha='right')
for bar, v in zip(bars, vals_d):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('10_discovery.png', bbox_inches='tight')
plt.show()

## 12. 吸引觀看原版的影視類型

In [ ]:
genre_raw = split_multi(df['genres_attract_original'])
genre_clean = [g for g in genre_raw if '其他' not in g and g]
genre_counts = Counter(genre_clean)

gc_items = sorted(genre_counts.items(), key=lambda x: -x[1])
fig, ax = plt.subplots(figsize=(10, 5))
labels_g = [k for k, v in gc_items]
vals_g = [v for k, v in gc_items]
ax.bar(labels_g, vals_g, color=sns.color_palette('tab10', len(labels_g)))
ax.set_title('吸引觀看原版的影視類型（複選）', fontsize=14)
ax.set_ylabel('提及次數')
for i, v in enumerate(vals_g):
    ax.text(i, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('11_genres.png', bbox_inches='tight')
plt.show()

## 13. 「整體影響」的個人層面分析（行為得分）

In [ ]:
# Score each person's actions_after_condensed
PROMOTE_ACTIONS = {
    '在已訂閱的串流平台看完完整作品',
    '在已訂閱的串流平台觀看作品部分內容',
    '透過正版免費管道觀看原版（如電視播映、YouTube 官方頻道）',
    '將原版加入待看清單',
}
ERODE_ACTIONS = {
    '至 YouTube 等平台找其他此作品的濃縮影片觀看',
    '透過非正式授權管道觀看原版（如盜版網站、非授權影片）',
}

def score_person(row):
    actions = [a.strip() for a in str(row.get('actions_after_condensed', '')).split('|') if a.strip()]
    promote = sum(1 for a in actions if a in PROMOTE_ACTIONS)
    erode = sum(1 for a in actions if a in ERODE_ACTIONS)
    return pd.Series({'promote_score': promote, 'erode_score': erode, 'net_score': promote - erode})

scores = df.apply(score_person, axis=1)
df_s = pd.concat([df[['gender', 'age', 'ntu_student', 'overall_impact_on_original']], scores], axis=1)

print('行為得分統計：')
print(df_s[['promote_score', 'erode_score', 'net_score']].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title, color in zip(axes,
    ['promote_score', 'erode_score', 'net_score'],
    ['促進行為數', '侵蝕行為數', '淨行為分（促進−侵蝕）'],
    ['#4CAF50', '#F44336', '#2196F3']):
    ax.hist(df_s[col], bins=range(int(df_s[col].min())-1, int(df_s[col].max())+2),
            color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('分數')
    ax.set_ylabel('人數')
    ax.axvline(df_s[col].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'平均 {df_s[col].mean():.2f}')
    ax.legend(fontsize=10)

plt.suptitle('個人行為得分分佈', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('12_behavior_scores.png', bbox_inches='tight')
plt.show()

## 14. 自答「整體影響」 vs 實際行為得分 交叉驗證

In [ ]:
impact_order = ['讓我看了更多原版作品', '視作品而定，沒有固定傾向',
                '沒有明顯影響', '讓我看了更少原版作品']
df_s2 = df_s.copy()
df_s2['impact'] = df_s2['overall_impact_on_original']
df_s2 = df_s2[df_s2['impact'].isin(impact_order)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, score_col, title, color in zip(axes,
    ['promote_score', 'net_score'],
    ['自述影響 vs 促進行為數', '自述影響 vs 淨行為分'],
    ['#4CAF50', '#2196F3']):
    groups = [df_s2[df_s2['impact'] == k][score_col].values for k in impact_order if k in df_s2['impact'].values]
    valid_labels = [k for k in impact_order if k in df_s2['impact'].values]
    ax.boxplot(groups, labels=valid_labels, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.5))
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('分數')
    ax.set_xticklabels(valid_labels, rotation=15, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('13_self_report_vs_behavior.png', bbox_inches='tight')
plt.show()

## 15. 關鍵摘要

In [ ]:
n = len(df)
impact_pct = df['overall_impact_on_original'].value_counts(normalize=True) * 100

promote_pct = impact_pct.get('讓我看了更多原版作品', 0)
erode_pct = impact_pct.get('讓我看了更少原版作品', 0)
neutral_pct = impact_pct.get('沒有明顯影響', 0)
depends_pct = impact_pct.get('視作品而定，沒有固定傾向', 0)

piracy_n = df['actions_after_condensed'].str.contains('非正式授權', na=False).sum()
watchlist_n = df['actions_after_condensed'].str.contains('待看清單', na=False).sum()
completed_n = df['actions_after_condensed'].str.contains('看完完整作品', na=False).sum()

print('='*55)
print('  濃縮影片對正版影視消費影響 — 關鍵數字')
print('='*55)
print(f'  有效樣本: {n} 人')
print()
print('  【整體影響 (自述)】')
print(f'  ✅ 看了更多原版:   {promote_pct:.1f}%')
print(f'  ❌ 看了更少原版:   {erode_pct:.1f}%')
print(f'  ➖ 沒有明顯影響:   {neutral_pct:.1f}%')
print(f'  🔀 視作品而定:     {depends_pct:.1f}%')
print()
print('  【具體後續行為 (複選)】')
print(f'  📋 加入待看清單:    {watchlist_n} 人 ({watchlist_n/n*100:.1f}%)')
print(f'  ✅ 看完完整版:      {completed_n} 人 ({completed_n/n*100:.1f}%)')
print(f'  🏴‍☠️ 透過盜版觀看:   {piracy_n} 人 ({piracy_n/n*100:.1f}%)')
print()
avg_net = df_s['net_score'].mean()
print(f'  平均淨行為分（促進−侵蝕）: {avg_net:.2f}')
print(f'  → {"整體偏向促進" if avg_net > 0 else "整體偏向侵蝕" if avg_net < 0 else "整體中立"}')
print('='*55)

---
## 附錄：資料欄位說明

| 欄位 | 說明 |
|------|------|
| `screening_watched_condensed` | 篩選題：是否曾觀看濃縮影片 |
| `gender` | 性別 |
| `age` | 年齡層 |
| `ntu_student` | 是否為臺大學生 |
| `favorite_condensed_types` | 喜愛的濃縮影片類型（複選）|
| `watch_motivation` | 觀看動機（複選）|
| `actions_after_condensed` | 觀看後的行為（複選）|
| `low_effort_no_action_reasons` | 不進一步行動的原因 |
| `high_effort_watch_reasons` | 實際去看原版的原因 |
| `high_effort_watch_frequency` | 看完濃縮版後去看原版的頻率 |
| `discovered_via_condensed` | 透過濃縮片發現了哪些新內容 |
| `overall_impact_on_original` | 整體影響（自評）|
| `genres_attract_original` | 哪些類型吸引觀看原版 |